In [1]:
from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
pdf_path = '/Users/nitastha/Downloads/Fundamental Rules Hindi.pdf'

loader = PyPDFLoader(pdf_path)
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,
   chunk_overlap=120,
   length_function=len,
   is_separator_regex=False,)
final_documents = text_splitter.split_documents(documents)

In [41]:
from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
pdf_path = 'hinditextbook.pdf'

loader = PyPDFLoader(pdf_path)
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,
   chunk_overlap=50,
   length_function=len,
   is_separator_regex=False,)
final_documents = text_splitter.split_documents(documents)

In [42]:
documents

[Document(metadata={'source': 'hinditextbook.pdf', 'page': 0}, page_content='अध्या ् 2\nव्यावस याय्क संगठन क े  स वरूप\nअयिगम उद् ेश्\nइस अध्या ् क े  अध ््न क े  पश्\u200d यात् आप—\n yव्यावस याय्क संगठन क े  यवयिनन सवरूपों की पह ्\u200dयान कर सक ें गे;\n yव्यावस याय्क संगठनों क े  यवयिनन सवरूपों क े  लक्षण, ग ुण एव ं सीमयाओं को समझ सक ें गे; \n yयवयिनन व्यावस याय्क संगठनों क े  स वरूपों में अ ंतर कर सक ें गे; एव ं \n yव्यावस याय्क संगठन क े  उप ्ुक्त सवरूप क े  ्\u200d्न क े  यनरयाधारक तत वों की ्\u200d्\u200dयाधा कर सक ें गे।\nChapter-2.indd   29 25-05-2021   17:20:23\n2024-25\n'),
 Document(metadata={'source': 'hinditextbook.pdf', 'page': 1}, page_content='30\nव्यवसा ्य अध्य ्यन\n2.1 परिच ्\n्यि कोई व ्यक्त एक व ्वसया् प्यारंि करने  की \n्ोजनया बनया रहया है ्या वतधामयान व्वसया् कया यवसतयार \nकरनया ्\u200dयाहतया है, तो \nउसे स ंगठन क े  स वरूप क े  स ंबंर \nमें एक महत वपूणधा यनणधा्  लेनया होगया। सबसे उप ्ुक्त \nसवरूप क या यनरयाधारण करते सम ् व्यक्त को अपने \nसयारनों को ध\n्यान में रख

In [36]:
import pymupdf4llm

md_text = pymupdf4llm.to_markdown('hinditextbook.pdf')

# now work with the markdown text, e.g. store as a UTF8-encoded file
import pathlib
# for i in range(0, 10):
#     print(f"Page {i}:\n{md_text[i]}\n")
#     pathlib.Path(f"output{i}.md").write_bytes(md_text[i]['text'].encode())
pathlib.Path("output.md").write_bytes(md_text.encode())


Processing hinditextbook.pdf...
[                                        ] (0/3[=                                       ] ( 1/3[==                                      ] ( 2/34[===                                     ] ( 3/3[====                                    ] ( 4/34[=====                                   ] ( 5/3=[=======                                 ] ( 6/3[========                                ] ( 7/34[=========                               ] ( 8/3[==========                              ] ( 9/34[===========                             ] (10/3[============                            ] (11/34=[==============                          ] (12/34[===============                         ] (13/3[================                        ] (14/34[=================                       ] (15/3[==================                      ] (16/34=[====================                    ] (17/34[=====================                   ] (18/3[======================                  ] (1

180381

In [43]:
import fasttext as ft

# # # Download the FastText model
# !wget https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.hi.zip
# !unzip wiki.hi.zip

# Load the FastText model
embedding_model_path = '/Users/nitastha/Desktop/NitishFiles/Projects/wiki.hi/wiki.hi.bin'
embed_model = ft.load_model(embedding_model_path)

In [44]:
import pandas as pd

# convert the documents to a dataframe
# This dataframe will be used to create the embeddings
# And later will be used to update the Qdrant Vector Database
docs = final_documents
data = []
for doc in docs:
   # Get the page content and metadata for each chunk
   # Meta data contains chunk source or file name
   row_data = {
       "page_content": doc.page_content,
       "metadata": doc.metadata
   }
   data.append(row_data)

df = pd.DataFrame(data)

# Replace the new line characters with space
df['page_content'] = df['page_content'].replace('\\n', ' ', regex=True)

# Create a unique id for each document.
# This id will be used to update the Qdrant Vector Database
df['id'] = range(1, len(df) + 1)

# Create a payload column in the dataframe
# This payload column includes the page content and metadata
# This payload will be used when LLM needs to answer a query
df['payload'] = df[['page_content', 'metadata']].to_dict(orient='records')

# Create embeddings for each chunk
# This embeddings will be used when doing a similarity search with the user query
df['embeddings'] = df['page_content'].apply(lambda x: (embed_model.get_sentence_vector(x)).tolist())


In [45]:
df.head(2)

,page_content,metadata,id,payload,embeddings
0,अध्या ् 2 व्यावस याय्क संगठन क े स वरूप अयिगम...,"{'source': 'hinditextbook.pdf', 'page': 0}",1,{'page_content': 'अध्या ् 2 व्यावस याय्क संगठन...,"[-0.019271308556199074, 0.05125105381011963, -..."
1,30 व्यवसा ्य अध्य ्यन 2.1 परिच ् ्यि कोई व ्यक...,"{'source': 'hinditextbook.pdf', 'page': 1}",2,{'page_content': '30 व्यवसा ्य अध्य ्यन 2.1 पर...,"[-0.020177962258458138, 0.039215337485075, -0...."


In [46]:
df['page_content'][0]

'अध्या ् 2 व्यावस याय्क संगठन क े  स वरूप अयिगम उद् ेश् इस अध्या ् क े  अध ््न क े  पश्\u200d यात् आप—  yव्यावस याय्क संगठन क े  यवयिनन सवरूपों की पह ्\u200dयान कर सक ें गे;  yव्यावस याय्क संगठनों क े  यवयिनन सवरूपों क े  लक्षण, ग ुण एव ं सीमयाओं को समझ सक ें गे;   yयवयिनन व्यावस याय्क संगठनों क े  स वरूपों में अ ंतर कर सक ें गे; एव ं   yव्यावस याय्क संगठन क े  उप ्ुक्त सवरूप क े  ्\u200d्न क े  यनरयाधारक तत वों की ्\u200d्\u200dयाधा कर सक ें गे। Chapter-2.indd   29 25-05-2021   17:20:23 2024-25'

In [47]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, Batch

# Create a QdrantClient object
host = 'localhost'
port = 6333
client = QdrantClient(host=host, port=port)


In [48]:
# delete the collection if it already exists
client.delete_collection(collection_name="my_collection")

# Create a fresh collection in Qdrant
client.recreate_collection(
  collection_name="my_collection",
  vectors_config=VectorParams(size=300, distance=Distance.COSINE),
)

# Update the Qdrant Vector Database with the embeddings
# We are updating the embeddings in batches
# Since the data is large, we will only update the first batch of size 4000
batch_size = 4000
client.upsert(
collection_name="my_collection",
points=Batch(
    ids=df['id'].to_list()[:batch_size],
    payloads=df['payload'][:batch_size],
    vectors=df['embeddings'].to_list()[:batch_size],
),
)

# Close the QdrantClient
client.close()

/var/folders/7q/9t1f98rn10x46qx3z1s8kvp00000gp/T/ipykernel_84566/2073668465.py:5: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


In [49]:
import mlflow
from qdrant_client import QdrantClient
mlflow.end_run()
mlflow_lgging = True

if mlflow_lgging:
   # set the experiment name in the mlflow
   mlflow.set_experiment("Hindi Chatbot")
   # start the mlflow run
   mlflow.start_run()

# load the Qdrant client from the same host and port
# this client will be used to interact with the Qdrant server
host = "localhost"
port = 6333
client = QdrantClient(host=host, port=port)

# log the parameters in the mlflow
if mlflow_lgging:
   mlflow.log_param("qdrant_host", host)
   mlflow.log_param("qdrant_port", port)

In [50]:
if mlflow_lgging:
   mlflow.log_param("embed_model_path", embedding_model_path)

In [51]:
from typing import List
from qdrant_client import QdrantClient
import fasttext as ft
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever

# Define a custom retriever class that uses Qdrant for document retrieval
# Since we're using FastText embeddings, we won't be able to use the default lanchain retriever, as it only supports HuggingFace and OpenAI Models
class QdrantRetriever(BaseRetriever):
   client: QdrantClient
   embed_model: ft.FastText._FastText
   collection_name: str
   limit: int

   def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun) -> List[Document]:
       """Converts query to a vector and retrieves relevant documents using Qdrant."""
       # Get the vector representation of the query using the FastText model
       query_vector = self.embed_model.get_sentence_vector(query).tolist()

       # Search for the most similar documents in the Qdrant collection
       # The search method returns a list of hits, where each hit contains the most similar document
       # we can limit the number of hits to return using the limit parameter
       search_results = self.client.search(
           collection_name=self.collection_name,
           query_vector=query_vector,
           limit=self.limit
       )
       # Finally, we convert the search results to a list of Document objects
       # that can be used by the pipeline
       return [Document(page_content=hit.payload['page_content']) for hit in search_results]

collection_name="my_collection"
limit = 50

# use the Custom QdrantRetriever class to create a retriever object
retriever = QdrantRetriever(
   client=client,
   embed_model=embed_model,
   collection_name=collection_name,
   limit=limit
)

if mlflow_lgging:
   mlflow.log_param("collection_name", collection_name)
   mlflow.log_param("limit", limit)

In [25]:
# from langchain_community.llms.ollama import Ollama

# # Create an Ollama object with the specified parameters
# # This will very easily load the llama3 8-B model without the need of separately handling tokenizer like we do in huggingface
# model_name = 'llama3'
# num_predict = 100
# num_ctx = 3000
# num_gpu = 2
# temperature = 0.7
# top_k = 50
# top_p = 0.95


# llm=Ollama(model=model_name, num_predict=num_predict, num_ctx=num_ctx, num_gpu=num_gpu, temperature=temperature, top_k=top_k, top_p=top_p)



# if mlflow_lgging:
#    mlflow.log_param("model_name", model_name)
#    mlflow.log_param("num_predict", num_predict)
#    mlflow.log_param("num_ctx", num_ctx)
#    mlflow.log_param("num_gpu", num_gpu)
#    mlflow.log_param("temperature", temperature)
#    mlflow.log_param("top_k", top_k)
#    mlflow.log_param("top_p", top_p)

: 

In [13]:
from langchain_core.prompts import ChatPromptTemplate

# system_prompt = (
#    """<s>[INST] आप एक सम्मानीय सहायक हैं। आपका काम नीचे दिए गए संदर्भ से प्रश्नों का उत्तर देना है। आप केवल हिंदी भाषा में उत्तर दे सकते हैं। धन्यवाद।
#    You are never ever going to generate responses in English. You are always going to generate responses in Hindi no matter what. You also need to keep your answer short and to the point.

#    संदर्भ: {context} </s>
# """
# )

# prompt = ChatPromptTemplate.from_messages(
#    [
#        ("system", system_prompt),
#        ("human", "{input}"),
#    ]
# )

# if mlflow_lgging:
#    mlflow.log_param("system_prompt", system_prompt)

In [52]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# # Create a chain that combines the retriever and the question-answer chain
# # essentially, this chain will retrieve relevant documents using the retriever
# # and the prompts

# question_answer_chain = create_stuff_documents_chain(llm, prompt)
# chain = create_retrieval_chain(retriever, question_answer_chain)

In [29]:
# query = 'प्रारंभिक वेतन निर्धारण करने का अधिकार किसे है?'

# if mlflow_lgging:
#    mlflow.log_param("query", query)

# response = chain.invoke({"input": query})

# if mlflow_lgging:
#    mlflow.log_param("context", response['context'])
#    mlflow.log_param("response", response['answer'])

# print(response)

# # end the logging of the mlflow
# mlflow.end_run()

{'input': 'प्रारंभिक वेतन निर्धारण करने का अधिकार किसे है?', 'context': [Document(page_content='िद का ननमागण करने हेतु सक्षम प्राधधकारी की मंजूरी के बबना उसके िद के र्लए मंजूर िेतन से अधधक हो जाए'), Document(page_content='रूि से अनुिक्स्थत रहते है, को विभागीय जाँच के दौरान ननलंबन में रखना आिश्यक नहीं है, तयोंकक ऐसा करने'), Document(page_content='6 माह तक आधा िेतन और उसके बाद तीन-चौथाई िेतन का भुगतान करना िडता है । अनाधधकृत रूि से अनुिक्स्थत'), Document(page_content='ऐसा करने से िे ननलंबन भत्ते आदद की माँग करते हैं ।  3. सक्षम प्राधधकारी सुननक्श्चत करें कक लंबी'), Document(page_content='ननलंबन भत्ते की माँग करता है । यदद ऐसे प्रकरणों में ककसी शासकीय सेिक को ननलंबन में नहीं रखा जाए तो')], 'answer': 'सक्षम प्राधधकारी को।'}


: 

In [34]:
# response

{'input': 'प्रारंभिक वेतन निर्धारण करने का अधिकार किसे है?',
 'context': [Document(page_content='िद का ननमागण करने हेतु सक्षम प्राधधकारी की मंजूरी के बबना उसके िद के र्लए मंजूर िेतन से अधधक हो जाए'),
  Document(page_content='रूि से अनुिक्स्थत रहते है, को विभागीय जाँच के दौरान ननलंबन में रखना आिश्यक नहीं है, तयोंकक ऐसा करने'),
  Document(page_content='6 माह तक आधा िेतन और उसके बाद तीन-चौथाई िेतन का भुगतान करना िडता है । अनाधधकृत रूि से अनुिक्स्थत'),
  Document(page_content='ऐसा करने से िे ननलंबन भत्ते आदद की माँग करते हैं ।  3. सक्षम प्राधधकारी सुननक्श्चत करें कक लंबी'),
  Document(page_content='ननलंबन भत्ते की माँग करता है । यदद ऐसे प्रकरणों में ककसी शासकीय सेिक को ननलंबन में नहीं रखा जाए तो')],
 'answer': 'सक्षम प्राधधकारी को।'}

: 

In [37]:

# query = 'राज्य सचिव आदेश क्या है?'

# if mlflow_lgging:
#    mlflow.log_param("query", query)

# response = chain.invoke({"input": query})

# if mlflow_lgging:
#    mlflow.log_param("context", response['context'])
#    mlflow.log_param("response", response['answer'])

# print(response)

# # end the logging of the mlflow
# mlflow.end_run()

{'input': 'राज्य सचिव आदेश क्या है?', 'context': [Document(page_content='िररषद के राज्यिाल की शक्ततयों िर आदेश द्िारा अधधरोवित ककन्हीं प्रनतबंधों के अधीन रहते, जैसा भी'), Document(page_content='(ब) यदद राज्य सरकार राज्यिाल के प्रान्त की स्थानीय सरकार है, तो ऐसा विशेष  िेतन अथिा व्यक्ततगत िेतन'), Document(page_content='।  िाज्य सधचि आदेश 1 - उन मामलों में जहाँ राज्य शासन ने भारतीय र्सविल सेिा के अधधकाररयों को मूल'), Document(page_content='िद का ननमागण करने हेतु सक्षम प्राधधकारी की मंजूरी के बबना उसके िद के र्लए मंजूर िेतन से अधधक हो जाए'), Document(page_content='ननणगय र्लया जा चुका है- िूिग में डाइस-नान के प्रकरण वित्त विभाग को भेजे जाते थे । राज्य शासन ने')], 'answer': 'राज्य सचिव आदेश 1 है, जिसमें राज्य सरकार ने मूल्यांकन करने हेतु सक्षम प्राधधकारी की मंजूरी के बबना उसके नाम के र्लए मंजूर िेतन से अधधक हो जाए।'}


: 

### Working with GROQ API now

In [53]:
import config
from langchain_groq import ChatGroq
groq_api_key = config.GROQ_API_KEY
selected_model = "llama-3.1-70b-versatile"
llm = ChatGroq(groq_api_key=groq_api_key, model_name=selected_model)

# if mlflow_lgging:
#    mlflow.log_param("model_name", selected_model)

from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    """<s>[INST] आप एक विश्वसनीय और सटीक सहायक हैं। आपको केवल और केवल नीचे दिए गए संदर्भ के आधार पर प्रश्न का उत्तर देना है। 

निर्देश:
- केवल दिए गए संदर्भ से जानकारी का उपयोग करें
- यदि संदर्भ में उत्तर नहीं मिलता है, तो स्पष्ट रूप से कहें कि "दिए गए संदर्भ में इस प्रश्न का उत्तर नहीं मिलता"
- अपने ज्ञान या अतिरिक्त जानकारी को शामिल न करें
- उत्तर संक्षिप्त, स्पष्ट और सीधा होना चाहिए
- हिंदी भाषा में ही उत्तर दें

संदर्भ: {context} </s>
"""
)

prompt = ChatPromptTemplate.from_messages(
   [
       ("system", system_prompt),
       ("human", "{input}"),
   ]
)

if mlflow_lgging:
   mlflow.log_param("system_prompt", system_prompt)


question_answer_chain = create_stuff_documents_chain(llm, prompt)
chain = create_retrieval_chain(retriever, question_answer_chain)

In [54]:
query = 'प्रारंभिक वेतन निर्धारण करने का अधिकार किसे है?'

if mlflow_lgging:
   mlflow.log_param("query", query)

response = chain.invoke({"input": query})

if mlflow_lgging:
   mlflow.log_param("context", response['context'])
   mlflow.log_param("response", response['answer'])

print(response)

# end the logging of the mlflow
mlflow.end_run()

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.1-70b-versatile` in organization `org_01hs1sp1bfftj9dhthfs18jhhf` on tokens per minute (TPM): Limit 6000, Requested 27646, please reduce your message size and try again. Visit https://console.groq.com/docs/rate-limits for more information.', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [63]:
response['context']

[Document(page_content='।  िाज्य सधचि आदेश 1 - उन मामलों में जहाँ राज्य शासन ने भारतीय र्सविल सेिा के अधधकाररयों को मूल'),
 Document(page_content='ननणगय र्लया जा चुका है- िूिग में डाइस-नान के प्रकरण वित्त विभाग को भेजे जाते थे । राज्य शासन ने'),
 Document(page_content='िररषद के राज्यिाल की शक्ततयों िर आदेश द्िारा अधधरोवित ककन्हीं प्रनतबंधों के अधीन रहते, जैसा भी'),
 Document(page_content='िद का ननमागण करने हेतु सक्षम प्राधधकारी की मंजूरी के बबना उसके िद के र्लए मंजूर िेतन से अधधक हो जाए'),
 Document(page_content='(ब) यदद राज्य सरकार राज्यिाल के प्रान्त की स्थानीय सरकार है, तो ऐसा विशेष  िेतन अथिा व्यक्ततगत िेतन'),
 Document(page_content='होगी ।   महालेखा पिीक्षक अिुदेश 1- नियम जो मूल नियम 22 एिं 23 को िद्द िहीं किेंगे- मूल ननयम 19 की'),
 Document(page_content='को मूल ननयम 19 (2) (v) के अन्तगगत व्यक्ततगत िेतन मंजूर ककया है, ताकक भारतीय र्सविल सेिा के समयमान'),
 Document(page_content='सहिदठत मूल ननयम 17-ए के अधीन सभी उद्देश्यों के र्लये सेिा में व्यिधान माना जािे । ऐसे सेिकों को'),
 Docu

: 

In [62]:

query = 'राज्य सचिव आदेश 1 क्या है?'

if mlflow_lgging:
   mlflow.log_param("query", query)

response = chain.invoke({"input": query})

if mlflow_lgging:
   mlflow.log_param("context", response['context'])
   mlflow.log_param("response", response['answer'])

print(response)

# end the logging of the mlflow
mlflow.end_run()

BadRequestError: Error code: 400 - {'error': {'message': 'Please reduce the length of the messages or completion.', 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}

: 

In [50]:
response

{'input': 'राज्य सचिव आदेश 1 क्या है?',
 'context': [Document(page_content='।  िाज्य सधचि आदेश 1 - उन मामलों में जहाँ राज्य शासन ने भारतीय र्सविल सेिा के अधधकाररयों को मूल'),
  Document(page_content='ननणगय र्लया जा चुका है- िूिग में डाइस-नान के प्रकरण वित्त विभाग को भेजे जाते थे । राज्य शासन ने'),
  Document(page_content='िररषद के राज्यिाल की शक्ततयों िर आदेश द्िारा अधधरोवित ककन्हीं प्रनतबंधों के अधीन रहते, जैसा भी'),
  Document(page_content='िद का ननमागण करने हेतु सक्षम प्राधधकारी की मंजूरी के बबना उसके िद के र्लए मंजूर िेतन से अधधक हो जाए'),
  Document(page_content='(ब) यदद राज्य सरकार राज्यिाल के प्रान्त की स्थानीय सरकार है, तो ऐसा विशेष  िेतन अथिा व्यक्ततगत िेतन'),
  Document(page_content='होगी ।   महालेखा पिीक्षक अिुदेश 1- नियम जो मूल नियम 22 एिं 23 को िद्द िहीं किेंगे- मूल ननयम 19 की'),
  Document(page_content='को मूल ननयम 19 (2) (v) के अन्तगगत व्यक्ततगत िेतन मंजूर ककया है, ताकक भारतीय र्सविल सेिा के समयमान'),
  Document(page_content='सहिदठत मूल ननयम 17-ए के अधीन सभी उद्देश्यों क

: 

### gemini api

In [55]:
from langchain_google_genai import ChatGoogleGenerativeAI
import config


# Set up Gemini API
model_name = config.CHAT_MODEL  # Gemini Pro model
google_api_key = config.GOOGLE_API_KEY  # Replace with your actual Google API key

llm = ChatGoogleGenerativeAI(
    model=model_name,
    google_api_key=google_api_key,
    temperature=0.7,
    max_output_tokens=100
)

from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    """<s>[INST] आप एक विश्वसनीय और सटीक सहायक हैं। आपको केवल और केवल नीचे दिए गए संदर्भ के आधार पर प्रश्न का उत्तर देना है। 

निर्देश:
- केवल दिए गए संदर्भ से जानकारी का उपयोग करें
- यदि संदर्भ में उत्तर नहीं मिलता है, तो स्पष्ट रूप से कहें कि "दिए गए संदर्भ में इस प्रश्न का उत्तर नहीं मिलता"
- अपने ज्ञान या अतिरिक्त जानकारी को शामिल न करें
- उत्तर संक्षिप्त, स्पष्ट और सीधा होना चाहिए
- हिंदी भाषा में ही उत्तर दें

संदर्भ: {context} </s>
"""
)

prompt = ChatPromptTemplate.from_messages(
   [
       ("system", system_prompt),
       ("human", "{input}"),
   ]
)

if mlflow_lgging:
   mlflow.log_param("system_prompt", system_prompt)


question_answer_chain = create_stuff_documents_chain(llm, prompt)
chain = create_retrieval_chain(retriever, question_answer_chain)

In [58]:

query = 'जे.एल. हंसन ने क्या कहा है?'

# if mlflow_lgging:
#    mlflow.log_param("query", query)

response = chain.invoke({"input": query})

# if mlflow_lgging:
#    mlflow.log_param("context", response['context'])
#    mlflow.log_param("response", response['answer'])

print(response)

# end the logging of the mlflow
# mlflow.end_run()

{'input': 'जे.एल. हंसन ने क्या कहा है?', 'context': [Document(page_content='संबंयरत यनणधा्  लेन े की बह ुत अयरक सवतंत्रतया  होती है क््ोंयक उसे यकसी िूसरे से सल याह की  आवश ्कतया नहीं है  इसयलए वह त ुरंत यनणधा्   ले सकत या है। इसक े  कयारण जब िी उसे कोई  स्फूय\u200dतमादया्क प्यािंभ— कोक या-कोल या प्यािंभ में एकल स वयायमतव कया व्वसया् थया! िुयन्याि र को एक ख यास सवयाि से पर रय्\u200dत करव याने वयाले कोक या-कोल या की श ुरूआत 8  मई 1886 को ए िलयांिया,  जॉयजधा्या से हुई थी।  डॉ. जॉन यसमथ पैंब िधान एक स थयानी् औषयर यनमयाधातया थे। उन होंने कोक या-कोल या क े  नयाम से  एक श बधात बनया्या। वे इस नए  उतपयाि को एक प यास में यसथत जैकब फ याममेसी में ल े गए। वह याँ उसक या नमूनया ्\u200dखया  ग्या तथया उसे अद् ुत घोयषत यक्या ग्या। एक सोड या पे् क े  रूप  में वह प याँ्\u200d सैंि प्यत यगलयास बे्\u200dया जयाने लग या।  पैंबिधान को अपने उत पयाि की यनयहत संियावनयाओं कया अहस यास िी नहीं ह ुआ।  उनहोंने रीरे-रीरे अपने  व्वसया्  को िुकडों में अपने  सयाझेियारों को बे ्\u200d यि्या और 1888 म ें अपनी म ृत्ु क े  क

In [59]:
response

{'input': 'जे.एल. हंसन ने क्या कहा है?',
 'context': [Document(page_content='संबंयरत यनणधा्  लेन े की बह ुत अयरक सवतंत्रतया  होती है क््ोंयक उसे यकसी िूसरे से सल याह की  आवश ्कतया नहीं है  इसयलए वह त ुरंत यनणधा्   ले सकत या है। इसक े  कयारण जब िी उसे कोई  स्फूय\u200dतमादया्क प्यािंभ— कोक या-कोल या प्यािंभ में एकल स वयायमतव कया व्वसया् थया! िुयन्याि र को एक ख यास सवयाि से पर रय्\u200dत करव याने वयाले कोक या-कोल या की श ुरूआत 8  मई 1886 को ए िलयांिया,  जॉयजधा्या से हुई थी।  डॉ. जॉन यसमथ पैंब िधान एक स थयानी् औषयर यनमयाधातया थे। उन होंने कोक या-कोल या क े  नयाम से  एक श बधात बनया्या। वे इस नए  उतपयाि को एक प यास में यसथत जैकब फ याममेसी में ल े गए। वह याँ उसक या नमूनया ्\u200dखया  ग्या तथया उसे अद् ुत घोयषत यक्या ग्या। एक सोड या पे् क े  रूप  में वह प याँ्\u200d सैंि प्यत यगलयास बे्\u200dया जयाने लग या।  पैंबिधान को अपने उत पयाि की यनयहत संियावनयाओं कया अहस यास िी नहीं ह ुआ।  उनहोंने रीरे-रीरे अपने  व्वसया्  को िुकडों में अपने  सयाझेियारों को बे ्\u200d यि्या और 1888 म ें अपनी म ृत्ु क े  

In [63]:
# query = 'एल.एच. हेनी ने क्या कहा है?'
query = 'असीयम‍त दयाय्त'
# if mlflow_lgging:
#    mlflow.log_param("query", query)

response = chain.invoke({"input": query})

# if mlflow_lgging:
#    mlflow.log_param("context", response['context'])
#    mlflow.log_param("response", response['answer'])
response

# end the logging of the mlflow
# mlflow.end_run()

{'input': 'असीयम\u200dत दयाय्त',
 'context': [Document(page_content='59 व्यावस याय्क संगठन क े  स वरूप सयाियांश व्वसया् संगठन क े  यवयभ नन सवरूप यनमन हैं— 1. एकल स वयायमतव 2. सं्ुक्त यहंिू पररवयार व्वसया् 3. सयाझेियारी   4. सहक यारी सयमयत तथया 5. सं्ुक्त पूँजी कंपनी एकल स वयायमतव एकल स वयायमतव उस व ्वसया् को कहते  हैं, यजसकया सवयायमतव, प्बंरन एवं यन्ंत्रण एक ही व ्यक्त क े  हयाथ में होत या  है तथ या वही स ंपूणधा लयाि पयाने कया अयरकयारी तथ या हयायन क े  यलए उत्तर िया्ी होतया है। एकल  सवयायमतव क े  कई  लयाि हैं।  इनमें  से कुछ महत वपूणधा लयाि यनमन हैं— 1.  शीघधा यनणधा्  2. स ू्\u200dनया की गोपनी ्तया 3. प्त्क्ष प्ोतसयाहन 4. उपल यबर  कया अहस यास 5. स थयायपत करने एव ं  बंि करने म ें सुगमतया। उपरोक् त लयािों क े  होत े हुए िी एकल स वयायमतव की िी  कुछ सीम याएँ हैं।  इनमें से क ुछ प्मुख सीम याएँ इस प्कयार हैं— 1.  सीयमत संसयारन 2. व्यावस याय्क इक याई कया सीयमत  जीवनक याल 3. असी यमत ियाय्तव 4. सी यमत प्बंर ्ोग्तया।  सं्ु्\u200d\u200dत यहंदफू परिव याि व्वसया् इसकया अयिप्या् उस व ्वसया् से है यज

In [61]:
response

{'input': 'एल.एच. हेनी ने क्या कहा है?',
 'context': [Document(page_content='संबंयरत यनणधा्  लेन े की बह ुत अयरक सवतंत्रतया  होती है क््ोंयक उसे यकसी िूसरे से सल याह की  आवश ्कतया नहीं है  इसयलए वह त ुरंत यनणधा्   ले सकत या है। इसक े  कयारण जब िी उसे कोई  स्फूय\u200dतमादया्क प्यािंभ— कोक या-कोल या प्यािंभ में एकल स वयायमतव कया व्वसया् थया! िुयन्याि र को एक ख यास सवयाि से पर रय्\u200dत करव याने वयाले कोक या-कोल या की श ुरूआत 8  मई 1886 को ए िलयांिया,  जॉयजधा्या से हुई थी।  डॉ. जॉन यसमथ पैंब िधान एक स थयानी् औषयर यनमयाधातया थे। उन होंने कोक या-कोल या क े  नयाम से  एक श बधात बनया्या। वे इस नए  उतपयाि को एक प यास में यसथत जैकब फ याममेसी में ल े गए। वह याँ उसक या नमूनया ्\u200dखया  ग्या तथया उसे अद् ुत घोयषत यक्या ग्या। एक सोड या पे् क े  रूप  में वह प याँ्\u200d सैंि प्यत यगलयास बे्\u200dया जयाने लग या।  पैंबिधान को अपने उत पयाि की यनयहत संियावनयाओं कया अहस यास िी नहीं ह ुआ।  उनहोंने रीरे-रीरे अपने  व्वसया्  को िुकडों में अपने  सयाझेियारों को बे ्\u200d यि्या और 1888 म ें अपनी म ृत्ु क े  

In [62]:
query = 'कोक के बारे में क्या कहा गया है?'

# if mlflow_lgging:
#    mlflow.log_param("query", query)

response = chain.invoke({"input": query})

# if mlflow_lgging:
#    mlflow.log_param("context", response['context'])
#    mlflow.log_param("response", response['answer'])

response

# end the logging of the mlflow
# mlflow.end_run()

{'input': 'कोक के बारे में क्या कहा गया है?',
 'context': [Document(page_content='संबंयरत यनणधा्  लेन े की बह ुत अयरक सवतंत्रतया  होती है क््ोंयक उसे यकसी िूसरे से सल याह की  आवश ्कतया नहीं है  इसयलए वह त ुरंत यनणधा्   ले सकत या है। इसक े  कयारण जब िी उसे कोई  स्फूय\u200dतमादया्क प्यािंभ— कोक या-कोल या प्यािंभ में एकल स वयायमतव कया व्वसया् थया! िुयन्याि र को एक ख यास सवयाि से पर रय्\u200dत करव याने वयाले कोक या-कोल या की श ुरूआत 8  मई 1886 को ए िलयांिया,  जॉयजधा्या से हुई थी।  डॉ. जॉन यसमथ पैंब िधान एक स थयानी् औषयर यनमयाधातया थे। उन होंने कोक या-कोल या क े  नयाम से  एक श बधात बनया्या। वे इस नए  उतपयाि को एक प यास में यसथत जैकब फ याममेसी में ल े गए। वह याँ उसक या नमूनया ्\u200dखया  ग्या तथया उसे अद् ुत घोयषत यक्या ग्या। एक सोड या पे् क े  रूप  में वह प याँ्\u200d सैंि प्यत यगलयास बे्\u200dया जयाने लग या।  पैंबिधान को अपने उत पयाि की यनयहत संियावनयाओं कया अहस यास िी नहीं ह ुआ।  उनहोंने रीरे-रीरे अपने  व्वसया्  को िुकडों में अपने  सयाझेियारों को बे ्\u200d यि्या और 1888 म ें अपनी म ृत्ु 